<span style="color:#B8860B; font-weight:bold; font-size:30px;">
Import Library
</span>

In [7]:
import rby1_sdk as rby
from dynamixel_sdk import PortHandler, PacketHandler
import time
import numpy as np
import sys
from pathlib import Path

# 0: folder_a/folder_b-file.py -> folder_a / 1: folder_a/folder_b/folder_c-file.py -> folder_a
sys.path.append(str(Path.cwd().parents[0]))                         # .ipynb
# sys.path.append(str(Path(__file__).resolve().parents[0]))           # .py
import functions as fn

np.set_printoptions(suppress=True, precision=5, linewidth=300)  

SIM_OR_REAL = 0         # 0: Docker Simulation / 1: Real Robot(Wifi Remote Access mode) / 2: Real Robot (UPC Direct mode)

<span style="color:#B8860B; font-weight:bold; font-size:30px;">
Power ON
</span>

In [4]:
if (SIM_OR_REAL == 0):
    ROBOT_ADDRESS = "localhost:50051"           # For Docker Simulation
elif (SIM_OR_REAL == 1 or SIM_OR_REAL == 2):
    ROBOT_ADDRESS = "192.168.30.1:50051"        # For Real Robot
else:
    ROBOT_ADDRESS = "localhost:50051"           # For Safety, Default=Docker Simulation

robot = rby.create_robot_a(ROBOT_ADDRESS)       # Robot Model is "A"
model = robot.model()
if not robot.connect():
    print("Could not connect to robot")
    exit(1)
else:
    print("SUCCESS: Connect to the robot.")
    if (robot.power_on(".*") == True):             # Turn on all the power (5v, 12v, 24v, 48v)
        time.sleep(1)
        if (robot.servo_on(".*") == True):         # Enable servo motors on all actuators
            time.sleep(1)
            robot.reset_fault_control_manager()
            robot.enable_control_manager(unlimited_mode_enabled=True)
            print("SUCCESS: Power ON the Robot.")
            
            # Grippers 12V Power ON 
            time.sleep(1)
            robot.set_tool_flange_output_voltage("right", 12)
            robot.set_tool_flange_output_voltage("left", 12)
            print("SUCCESS: Power ON the Gripper.")
        else:
            print("Failed to power ON the Servo")

SUCCESS: Connect to the robot.
SUCCESS: Power ON the Robot.
SUCCESS: Power ON the Gripper.


<span style="color:#B8860B; font-weight:bold; font-size:30px;">
Power OFF
</span>

In [3]:
robot.set_tool_flange_output_voltage("right", 0)
robot.set_tool_flange_output_voltage("left", 0)
time.sleep(1)
robot.disable_control_manager()
robot.power_off(".*")

True

<span style="color:#5ea6ff; font-weight:bold; font-size:30px;">
Test for Robot State
</span>

In [ ]:
import inspect

robot_state = robot.get_state()
robot_state_properties = [name for name, obj in
                          inspect.getmembers(type(robot_state), lambda x: isinstance(x, property))]
print(f"Available Robot States:\n {robot_state_properties}")

<span style="color:#B8860B; font-weight:bold; font-size:30px;">
Posture Change
</span>

In [5]:
Desired_Pos = fn.ZERO_POS                             # ZERO_POS, READY_POS, PACK_POS, READY_DEMO_POS
fn.move_position_22dof(robot, Desired_Pos, minimum_time=5.0, control_hold_time=0.5)
time.sleep(1)

<span style="color:#B8860B; font-weight:bold; font-size:30px;">
Functions
</span>

In [10]:
COLLISION_THRESHOLD = 0.05
stop_requested = False
RIGHT_ARM_TARGET = np.array([0.8, -0.8, 0.7, -2.0, 0, 0.0, 0])

def move_pre_control_pose(robot):
    """Move to the pre-control pose before starting the motion."""
    pose = {"torso":     np.array([0.0, 0.1, -0.2, 0.1, 0.0, 0.0]),
            "right_arm": np.array([0.2, -0.2, 0.0, -1.0, 0, 0.7, 0.0]),
            "left_arm":  np.array([0.2, 0.2, 0.0, -1.0, 0, 0.7, 0.0])}
    if not fn.move_position_20dof(robot, posture=pose, minimum_time=5.0, control_hold_time=0.5):
        exit(1)
        

def Callback_collision(robot_state):
    global stop_requested
    if not robot_state.collisions:
        return

    nearest = min(robot_state.collisions, key=lambda collision: collision.distance)
    print(
        f"nearest collision distance: {nearest.distance:.4f} | "
        f"{nearest.link1} <-> {nearest.link2}"
    )

    if nearest.distance < COLLISION_THRESHOLD:
        print(f"Collision Threshold Detected")
        stop_requested = True
        

def collision_test(robot):
    global stop_requested
    move_pre_control_pose(robot)
    stop_requested = False
    
    robot.start_state_update(Callback_collision, rate=50)
    
    collision_pose = {"torso":     None,
                      "right_arm": RIGHT_ARM_TARGET,
                      "left_arm":  None}
    rc = fn.move_position_20dof(robot, posture=collision_pose, minimum_time=5.0, control_hold_time=0.5)
    
    while not stop_requested:
        time.sleep(0.001)

    robot.cancel_control()
    robot.stop_state_update()
    

In [11]:
collision_test(robot)

nearest collision distance: 0.1430 | link_torso_4 <-> link_right_arm_3
nearest collision distance: 0.1430 | link_torso_4 <-> link_right_arm_3
nearest collision distance: 0.1430 | link_left_arm_3 <-> link_torso_4
nearest collision distance: 0.1430 | link_left_arm_3 <-> link_torso_4
nearest collision distance: 0.1430 | link_left_arm_3 <-> link_torso_4
nearest collision distance: 0.1430 | link_left_arm_3 <-> link_torso_4
nearest collision distance: 0.1430 | link_left_arm_3 <-> link_torso_4
nearest collision distance: 0.1430 | link_left_arm_3 <-> link_torso_4
nearest collision distance: 0.1430 | link_left_arm_3 <-> link_torso_4
nearest collision distance: 0.1430 | link_left_arm_3 <-> link_torso_4
nearest collision distance: 0.1430 | link_left_arm_3 <-> link_torso_4
nearest collision distance: 0.1430 | link_left_arm_3 <-> link_torso_4
nearest collision distance: 0.1430 | link_left_arm_3 <-> link_torso_4
nearest collision distance: 0.1430 | link_left_arm_3 <-> link_torso_4
nearest collision 